# CAL Agent with Context7 MCP Server

This notebook demonstrates how to create a CAL agent that connects to the
[Context7](https://context7.com) MCP server for up-to-date library documentation lookups.

### How it works

```
                          CAL Agent
                     +-----------------+
  User Query ------> |   GeminiLLM     |
                     |   + Memory      |
                     +--------+--------+
                              |
              +---------------+---------------+
              |               |               |
       +------+------+ +-----+------+ +------+------+
       | resolve-    | | query-docs | |    stop     |
       | library-id  | | (MCP Tool) | | (CAL Tool)  |
       | (MCP Tool)  | +-----+------+ +-------------+
       +------+------+       |
              |               |
              +-------+-------+
                      |
               +------+------+
               |  Context7   |
               |  MCP Server |
               |  (subprocess)|
               +-------------+
```

### Prerequisites

1. **Node.js / npx** — Context7 runs as an MCP server via `npx`
2. **Gemini API key** — set `GEMINI_API_KEY` in a `.env` file or as an environment variable
3. **Install CAL with MCP support:**

```bash
pip install "creevo-agent-library[mcp] @ git+https://github.com/Creevo-App/creevo-agent-library.git"
```

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install "creevo-agent-library[mcp] @ git+https://github.com/Creevo-App/creevo-agent-library.git"

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads GEMINI_API_KEY from .env

# Or set it directly:
# os.environ["GEMINI_API_KEY"] = "your-key-here"

---
## 1. Connect to the Context7 MCP Server

`connect_mcp_server` starts the MCP server as a subprocess and returns a list of
`MCPTool` instances — one per tool the server exposes.

In [ ]:
from CAL.mcp import connect_mcp_server, disconnect_mcp_tools

mcp_tools = await connect_mcp_server(
    command="npx",
    args=["-y", "@upstash/context7-mcp"],
)

print(f"Connected! Discovered {len(mcp_tools)} tools:\n")
for t in mcp_tools:
    print(f"  {t.name}")
    print(f"    {t.description[:100]}")
    print()

### Inspect Tool Schemas

Each `MCPTool` exposes a JSON schema describing its parameters — the same
format used by `@tool` and `@subagent` tools. Schemas from MCP servers are
automatically sanitized for Gemini compatibility.

In [ ]:
import json

for t in mcp_tools:
    schema = t.get_schema()
    print(f"--- {schema['name']} ---")
    print(json.dumps(schema["input_schema"], indent=2))
    print()

---
## 2. Create the Agent

Pass the MCP tools alongside `StopTool` into a standard CAL `Agent`.
No special configuration is needed — `MCPTool` implements the same `Tool` interface
as `@tool` and `@subagent`.

In [ ]:
from CAL import Agent, GeminiLLM, StopTool, FullCompressionMemory

api_key = os.getenv("GEMINI_API_KEY")

llm = GeminiLLM(model="gemini-2.5-flash", api_key=api_key, max_tokens=4096)
summarizer_llm = GeminiLLM(model="gemini-2.5-flash", api_key=api_key, max_tokens=2048)

agent = Agent(
    llm=llm,
    system_prompt=(
        "You are a helpful coding assistant. "
        "Use the Context7 MCP tools to look up library documentation before answering. "
        "Always cite the library version you referenced. "
        "Call stop when you have a complete answer."
    ),
    max_calls=15,
    max_tokens=4096,
    memory=FullCompressionMemory(summarizer_llm=summarizer_llm, max_tokens=50000),
    agent_name="context7-agent",
    tools=[StopTool(), *mcp_tools],
)

print(f"Agent ready with {len(agent.tools)} tools:")
for t in agent.tools:
    print(f"  - {t.name}")

---
## 3. Run a Query

The agent will:
1. Call `resolve-library-id` to find the Context7 ID for React
2. Call `query-docs` to fetch relevant documentation
3. Synthesize an answer and call `stop`

In [ ]:
from CAL.content_blocks import TextBlock, ToolUseBlock, ToolResultBlock
from CAL.message import MessageRole


def print_agent_response(agent):
    """Extract and print the agent's text response from conversation history."""
    # Collect all assistant text blocks (the response is in messages before 'stop')
    text_parts = []
    for msg in agent.memory.get_history():
        if msg.role == MessageRole.ASSISTANT and isinstance(msg.content, list):
            for block in msg.content:
                if isinstance(block, TextBlock) and block.text.strip():
                    text_parts.append(block.text)
    # Print the last text response (most recent answer)
    if text_parts:
        print("\n".join(text_parts[-2:]))  # last 2 parts cover explanation + code
    else:
        print("(No text response found)")


def print_tool_trace(agent):
    """Print a visual trace of the agent's tool calls."""
    print("Tool call trace:")
    step = 1
    for msg in agent.memory.get_history():
        if isinstance(msg.content, list):
            for block in msg.content:
                if isinstance(block, ToolUseBlock):
                    args_preview = json.dumps(block.input)[:80]
                    print(f"  [{step}] {block.name}({args_preview})")
                    step += 1
    print()

In [ ]:
result = await agent.run_async(
    "What does React useEffect do? Give a brief explanation with a small code example."
)

print_tool_trace(agent)
print("=" * 60)
print_agent_response(agent)

---
## 4. Try Another Query

The agent retains conversation history via `FullCompressionMemory`, so
follow-up questions work naturally without re-explaining context.

In [ ]:
result = await agent.run_async(
    "Now show me how to use useEffect with a cleanup function."
)

print_tool_trace(agent)
print("=" * 60)
print_agent_response(agent)

---
## 5. Conversation Summary

Visualize the full conversation flow — every message role and content type
that passed through the agent loop.

In [ ]:
ROLE_ICONS = {
    MessageRole.USER: "USER  ",
    MessageRole.ASSISTANT: "AGENT ",
}

print("Full conversation trace:\n")
for msg in agent.memory.get_history():
    icon = ROLE_ICONS.get(msg.role, "???   ")
    if isinstance(msg.content, str):
        print(f"  {icon} | {msg.content[:100]}")
    elif isinstance(msg.content, list):
        for block in msg.content:
            if isinstance(block, TextBlock):
                preview = block.text[:80].replace("\n", " ")
                print(f"  {icon} | text: {preview}..." if len(block.text) > 80 else f"  {icon} | text: {preview}")
            elif isinstance(block, ToolUseBlock):
                print(f"  {icon} | call: {block.name}({json.dumps(block.input)[:60]})")
            elif isinstance(block, ToolResultBlock):
                status = "error" if block.is_error else "ok"
                print(f"  {icon} | result({block.name}): [{status}]")

---
## 6. Cleanup

Always disconnect MCP tools when done to terminate the server subprocess.

In [ ]:
await disconnect_mcp_tools(mcp_tools)
print("MCP server disconnected.")